# 📦 Template 1 : préparation des données — AT&T Spam Detector

*Préparé par Claude le 19/10/2026 pour le projet `01-AT&T_spam_detector.ipynb`. Aucun code de solution ici : c'est toi qui écris.*

## Les 4 templates du projet

| # | Phase | Entrée | Sortie |
|---|---|---|---|
| **1** | **Préparation des données** (ce notebook) | `src/spam.csv` | `src/train.csv`, `src/val.csv`, `src/test.csv` |
| 2 | Modèle simple : embeddings + petit réseau (M05-D05) | les 3 CSV | un modèle + ses scores |
| 3 | Transfer learning : modèle Hugging Face (M06-D02) | les 3 CSV | un modèle + ses scores |
| 4 | Évaluation et conclusion | les scores | le tableau final du livrable |

Les CSV de sortie servent de point de départ aux templates 2 et 3 : ce notebook-ci n'est à faire qu'**une fois**.

## Comment s'en servir

Chaque étape suit le même rythme :

1. **🎯 Objectif** : ce qu'on veut obtenir, et pourquoi ;
2. **🧰 Outils** : les fonctions utiles, leur rôle, leurs arguments importants ;
3. **✍️ À toi** : une cellule avec le plan en commentaires, que tu remplis ;
4. **✅ Contrôle** : une cellule déjà écrite qui vérifie tes chiffres. Lance-la : ✅ = c'est bon, `AssertionError` = relis l'étape.

Les noms de variables imposés (`df`, `df_clean`, `train_df`...) sont ceux qu'attendent les contrôles.

---
# Étape 0 — Les imports
Rien à écrire : lance la cellule.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split

DOSSIER_SRC = Path("src")       # le dossier où se trouve spam.csv (à côté de ce notebook)
SEED = 42                       # graine du hasard, la même partout
print("prêt")

prêt


---
# Étape 1 — Charger le fichier

🎯 Lire `src/spam.csv` dans un DataFrame `df`.

⚠️ Le fichier n'est **pas** en UTF-8 (l'encodage par défaut de pandas) : sans le bon encodage, `read_csv` plante avec `UnicodeDecodeError`. Il est en **latin-1**.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `pd.read_csv(chemin, encoding=...)` | lit un CSV dans un DataFrame | `encoding="latin-1"` |
| `.head()`, `.shape`, `.info()` | premier coup d'œil | — |

In [47]:
# ✍️ À toi
# 1. df = le fichier DOSSIER_SRC / "spam.csv" lu avec le bon encodage
# 2. affiche les 5 premières lignes, la forme, puis .info()
url = 'https://raw.githubusercontent.com/MaquiZard05/Projet_ATT_jehda/main/src/spam.csv'
df = pd.read_csv(url, encoding="latin-1")
print("Done...\n")
display(df.head())

Done...



,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives around here though",NaN,NaN,NaN


👀 **Observe** : combien de colonnes ? Que contiennent `v1` et `v2` ? Combien de valeurs non vides dans les 3 colonnes `Unnamed` ?

In [48]:
# ✅ Contrôle
assert df.shape == (5572, 5), f"forme attendue (5572, 5), obtenue {df.shape}"
assert list(df.columns[:2]) == ["v1", "v2"]
print("✅ étape 1")

✅ étape 1


---
# Étape 2 — Nettoyer

## 2.a Repérer les messages coupés

🎯 Les colonnes `Unnamed: 2`, `Unnamed: 3`, `Unnamed: 4` sont presque vides. Mais quand elles ne le sont pas, ce n'est **pas** du bruit : c'est **la fin d'un SMS**. Certains messages contiennent des guillemets mal fermés, et le CSV les a coupés à chaque virgule.

```
v1     v2                                   Unnamed: 2       Unnamed: 3
spam   "Your free ringtone ... FML"          " PO Box 5249"   " MK17 92H. 450Ppw 16\""
       └─────────── un seul SMS, coupé en 3 morceaux à chaque virgule ───────────┘
```

Si tu supprimes ces colonnes tout de suite, tu perds la fin de ces messages.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `df[[col1, col2, col3]]` | garder plusieurs colonnes (double crochet = liste) | — |
| `.notna()` | `True` là où il y a une valeur | — |
| `.any(axis=1)` | `True` si **au moins une** case de la **ligne** est `True` | `axis=1` = ligne par ligne |
| `.sum()` | compte les `True` | — |

In [49]:
# ✍️ À toi
# 1. cols_en_trop = la liste des 3 noms de colonnes "Unnamed"
# 2. masque_coupes = True pour chaque ligne qui a au moins une valeur dans ces 3 colonnes
# 3. affiche le nombre de messages coupés, puis 3 de ces lignes pour les regarder
# Afficher les colonnes du dataset
print(df.columns.tolist())

# Compter les valeurs présentes dans les colonnes supplémentaires
extra_cols = ["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"]

print(df[extra_cols].notna().sum())


['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']
Unnamed: 2    50
Unnamed: 3    12
Unnamed: 4     6
dtype: int64


In [51]:

# Repérer les lignes qui contiennent au moins un morceau supplémentaire
mask = df[extra_cols].notna().any(axis=1)

print("Nombre de lignes concernées :", mask.sum())

# Afficher quelques exemples
display(df.loc[mask, ["v1", "v2"] + extra_cols].head(10))


Nombre de lignes concernées : 50


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
95,spam,"Your free ringtone is waiting to be collected. Simply text the password \MIX\"" to 85069 to verify. Get Usher and Britney. FML",PO Box 5249,"MK17 92H. 450Ppw 16""",NaN
281,ham,\Wen u miss someone,the person is definitely special for u..... But if the person is so special,why to miss them,"just Keep-in-touch\"" gdeve.."""
444,ham,\HEY HEY WERETHE MONKEESPEOPLE SAY WE MONKEYAROUND! HOWDY GORGEOUS,"HOWU DOIN? FOUNDURSELF A JOBYET SAUSAGE?LOVE JEN XXX\""""",NaN,NaN
671,spam,SMS. ac sun0819 posts HELLO:\You seem cool,"wanted to say hi. HI!!!\"" Stop? Send STOP to 62468""",NaN,NaN
710,ham,Height of Confidence: All the Aeronautics professors wer calld &amp; they wer askd 2 sit in an aeroplane. Aftr they sat they wer told dat the plane ws made by their students. Dey all hurried out of d plane.. Bt only 1 didnt move... He said:\if it is made by my students,"this wont even start........ Datz confidence..""",NaN,NaN
899,spam,"Your free ringtone is waiting to be collected. Simply text the password \MIX\"" to 85069 to verify. Get Usher and Britney. FML",PO Box 5249,"MK17 92H. 450Ppw 16""",NaN
1038,ham,"Edison has rightly said, \A fool can ask more questions than a wise man can answer\"" Now you know why all of us are speechless during ViVa.. GM",GN,GE,"GNT:-)"""
1127,ham,"Height of \Oh shit....!!\"" situation: A guy throws a luv letter on a gal but falls on her brothers head whos a gay",".;-):-D""",NaN,NaN
1266,ham,\Hey sorry I didntgive ya a a bellearlier hunny,"just been in bedbut mite go 2 thepub l8tr if uwana mt up?loads a luv Jenxxx.\""""",NaN,NaN
1384,ham,"Storming msg: Wen u lift d phne, u say \HELLO\"" Do u knw wt is d real meaning of HELLO?? . . . It's d name of a girl..! . . . Yes.. And u knw who is dat girl?? \""Margaret Hello\"" She is d girlfrnd f Grahmbell who invnted telphone... . . . . Moral:One can 4get d name of a person","bt not his girlfrnd... G o o d n i g h t . . .@""",NaN,NaN


In [52]:
# ✅ Contrôle
assert mask.sum() == 50, f"50 messages coupés attendus, obtenu {mask.sum()}"
print("✅ étape 2.a")

✅ étape 2.a


## 2.b Recoller les morceaux

🎯 Créer une colonne `text` = `v2` + les morceaux des colonnes `Unnamed`, **recollés avec une virgule** (la virgule que le CSV a prise pour un séparateur). Pour les 5 522 messages entiers, `text` = `v2`.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `",".join(morceaux)` | colle des textes avec une virgule entre eux | les morceaux doivent être des **str** (pas de `NaN`) |
| `.dropna()` sur **une ligne** | enlève les cases vides de cette ligne | — |
| `.apply(fonction, axis=1)` | applique une fonction à **chaque ligne** | `axis=1` |

Démo des deux premiers outils **sur une seule ligne**, dans un autre contexte (une adresse coupée) :

In [53]:
adresse = pd.Series(["12 rue des Lilas", " Nantes", None, None])   # une ligne avec des cases vides
print(adresse.dropna().tolist())          # les cases vides ont disparu
print(",".join(adresse.dropna()))         # recollé avec une virgule

['12 rue des Lilas', ' Nantes']
12 rue des Lilas, Nantes


Chez toi, c'est pareil, mais pour les 5 572 lignes : il faut appliquer ce geste à chaque ligne du DataFrame.

In [54]:
# ✍️ À toi
# 1. cols_texte = "v2" + les 3 colonnes Unnamed (dans cet ordre)
# 2. df["text"] = pour chaque ligne de df[cols_texte] : enlever les cases vides, recoller avec ","
# 3. affiche le text d'une ligne de masque_coupes pour vérifier
cols_texte = ["v2", "Unnamed: 2", "Unnamed: 3", "Unnamed: 4"]
df["text"] = df[cols_texte].apply(
    lambda ligne: ",".join(ligne.dropna().astype(str)),
    axis=1
)
display(df.loc[mask, "text"].head(1))
display(df.head())

95    Your free ringtone is waiting to be collected. Simply text the password \MIX\" to 85069 to verify. Get Usher and Britney. FML, PO Box 5249, MK17 92H. 450Ppw 16"
Name: text, dtype: str

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4,text
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",NaN,NaN,NaN,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,NaN,NaN,NaN,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives around here though",NaN,NaN,NaN,"Nah I don't think he goes to usf, he lives around here though"


In [55]:
# ✅ Contrôle
exemple = df.loc[df["v2"].str.startswith("Your free ringtone is waiting"), "text"].iloc[0]
assert exemple.endswith('MK17 92H. 450Ppw 16"'), f"la fin du message manque : {exemple!r}"
assert (df.loc[~mask, "text"] == df.loc[~mask, "v2"]).all(), "les messages entiers doivent rester identiques à v2"
print("✅ étape 2.b")

✅ étape 2.b


## 2.c Garder deux colonnes propres

🎯 Fabriquer `df_clean` avec **deux colonnes seulement** :
- `text` : le SMS recollé ;
- `label` : **0** pour `ham`, **1** pour `spam` (le modèle veut des nombres ; 1 = la classe qu'on cherche).

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `.map({ancienne: nouvelle, ...})` | remplace chaque valeur via un dictionnaire | les clés doivent couvrir **toutes** les valeurs, sinon `NaN` |
| `df[[...]].copy()` | nouveau DataFrame indépendant de `df` | — |

In [56]:
# ✍️ À toi
# 1. df["label"] = v1 traduit en 0 / 1
# 2. df_clean = une copie de df avec seulement les colonnes text et label (dans cet ordre)
# 3. affiche df_clean.head() et la répartition de label
# 1. Convertir les catégories en nombres
df["label"] = df["v1"].map({
    "ham": 0,
    "spam": 1
})

# 2. Garder uniquement les deux colonnes utiles
df_clean = df[["text", "label"]].copy()

# 3. Vérifier le résultat
display(df_clean.head())
print(df_clean.shape)
print(df_clean["label"].value_counts())
print(df_clean.isna().sum())

,text,label
0,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",0
1,Ok lar... Joking wif u oni...,0
2,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives around here though",0


(5572, 2)
label
0    4825
1     747
Name: count, dtype: int64
text     0
label    0
dtype: int64


In [57]:
# ✅ Contrôle
assert list(df_clean.columns) == ["text", "label"], f"colonnes obtenues : {list(df_clean.columns)}"
assert df_clean["label"].isna().sum() == 0, "des NaN dans label : la traduction a raté une valeur"
assert df_clean["label"].sum() == 747, f"747 spams attendus avant dédoublonnage, obtenu {df_clean['label'].sum()}"
print("✅ étape 2.c")

✅ étape 2.c


## 2.d Supprimer les doublons

🎯 Le même SMS apparaît plusieurs fois (spams envoyés en masse, « Ok lar... »). On garde **un exemplaire** de chaque texte.

❓ Question du matin : pourquoi le faire **avant** de séparer train et test ? Pense à ce qui arrive si un même SMS tombe dans les deux : le modèle passe l'examen avec une question qu'il a déjà vue en révision.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `.duplicated(subset=...)` | `True` pour chaque ligne déjà vue plus haut | `subset="text"` : on compare seulement le texte |
| `.drop_duplicates(subset=...)` | garde la première, supprime les suivantes | `subset="text"` |
| `.reset_index(drop=True)` | renumérote 0, 1, 2... | `drop=True` : jette l'ancien index |

In [58]:
# ✍️ À toi
# 1. affiche le nombre de doublons sur la colonne text
# 2. df_clean = df_clean sans doublons de text, index renuméroté
# 3. affiche la nouvelle forme et le nombre de spams
# 1. Compter les doublons de texte
print("Nombre de doublons :", df_clean.duplicated(subset="text").sum())

# 2. Supprimer les doublons
df_clean = df_clean.drop_duplicates(subset="text")

# 3. Réinitialiser les index
df_clean = df_clean.reset_index(drop=True)

# 4. Vérifier le résultat
print("Dimensions après nettoyage :", df_clean.shape)
print("Doublons restants :", df_clean.duplicated(subset="text").sum())

Nombre de doublons : 403
Dimensions après nettoyage : (5169, 2)
Doublons restants : 0


In [59]:
# ✅ Contrôle
assert df_clean.shape == (5169, 2), f"forme attendue (5169, 2), obtenue {df_clean.shape}"
assert df_clean["label"].sum() == 653, f"653 spams attendus, obtenu {df_clean['label'].sum()}"
assert df_clean.index.tolist() == list(range(5169)), "index pas renuméroté"
print("✅ étape 2.d")

✅ étape 2.d


---
# Étape 3 — Explorer

## 3.a Répartition des classes

🎯 Quelle part de spams ? C'est elle qui dit si l'accuracy est une bonne métrique ici.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `.value_counts(normalize=True)` | part de chaque valeur (entre 0 et 1) | `normalize=True` |
| `px.histogram(df, x=...)` | diagramme en barres d'une colonne | `x="label"` |

In [60]:
# ✍️ À toi
# 1. affiche la part de ham et de spam
# 2. trace un diagramme en barres de label
df_clean["label"].value_counts(normalize=True) * 100
part_spams = (df_clean["label"] == 1).mean() * 100
print(f"Part de spams : {part_spams:.2f} %")

Part de spams : 12.63 %


In [61]:
# ✅ Contrôle
assert round(df_clean["label"].mean(), 4) == 0.1263
print("✅ étape 3.a")

✅ étape 3.a


✍️ **À écrire (une phrase)** : un « modèle » qui répond **toujours ham** aurait quelle accuracy ? Qu'est-ce que ça dit de l'accuracy comme métrique pour ce projet ?

*Ta réponse :*
Les spams représentent 12,63 % du dataset, contre 87,37 % de messages normaux. Cette répartition déséquilibrée signifie qu'un modèle prédisant systématiquement la classe majoritaire pourrait atteindre 87,37 % d'accuracy sans détecter aucun spam.
On ne peut pas se fier uniquement à l'accuracy. On doit aussi mesurer combien de spams le modèle arrive réellement à détecter : c'est le recall.

## 3.b Longueur des messages

🎯 Comparer la longueur des ham et des spams, en caractères et en mots. Ça servira aussi en phase 2 pour choisir la longueur des séquences.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `.str.len()` | nombre de caractères de chaque texte | — |
| `.str.split().str.len()` | nombre de mots (on découpe aux espaces, puis on compte les morceaux) | — |
| `.groupby("label")[cols].median()` | médiane par classe | — |
| `px.histogram(df, x=..., color=...)` | deux histogrammes superposés | `color="label"`, `barmode="overlay"` |

In [62]:
# ✍️ À toi
# 1. df_clean["n_car"] = nombre de caractères ; df_clean["n_mots"] = nombre de mots
# 2. affiche la médiane de n_car et n_mots pour chaque classe
# 3. histogramme de n_mots, une couleur par classe
# Longueur en caractères
df_clean["nb_caracteres"] = df_clean["text"].str.len()

# Longueur en mots
df_clean["nb_mots"] = df_clean["text"].str.split().str.len()

display(df_clean.head())

,text,label,nb_caracteres,nb_mots
0,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",0,111,20
1,Ok lar... Joking wif u oni...,0,29,6
2,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,1,155,28
3,U dun say so early hor... U c already then say...,0,49,11
4,"Nah I don't think he goes to usf, he lives around here though",0,61,13


In [63]:
px.histogram(
    df_clean,
    x="nb_caracteres",
    color="label",
    barmode="overlay",
    title="Longueur des SMS en caractères selon la classe"
)

In [64]:
px.histogram(
    df_clean,
    x="nb_mots",
    color="label",
    barmode="overlay",
    title="Longueur des SMS en mots selon la classe"
)

In [22]:
# ✅ Contrôle
med = df_clean.groupby("label")[["nb_caracteres", "nb_mots"]].median()
assert med.loc[0, "nb_caracteres"] == 53 and med.loc[1, "nb_caracteres"] == 149, med
assert med.loc[0, "nb_mots"] == 11 and med.loc[1, "nb_mots"] == 25, med
print("✅ étape 3.b")

✅ étape 3.b


## 3.c Combien de mots garder par message ? (préparation de la phase 2)

🎯 En phase 2, chaque SMS deviendra une suite de numéros de **même longueur** (on coupe les trop longs, on complète les trop courts). Choisir une longueur `max_len` qui couvre presque tous les messages, sans s'aligner sur le plus long.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `.max()` | le plus long | — |
| `.quantile(q)` | la valeur sous laquelle se trouvent `q` % des messages | `q=0.95` |

In [65]:
# ✍️ À toi
# 1. affiche le nombre de mots du message le plus long
# 2. affiche le 95e centile (quantile 0.95) de n_mots
print(df_clean.max())
print(df_clean["nb_mots"].quantile(0.95))

text             ÌÏ wait 4 me in sch i finish ard 5..
label                                               1
nb_caracteres                                     910
nb_mots                                           171
dtype: object
32.599999999999454


In [66]:
# ✅ Contrôle
assert df_clean["nb_mots"].max() == 171
assert round(df_clean["nb_mots"].quantile(0.95)) == 33
print("✅ étape 3.c")

✅ étape 3.c


✍️ **À écrire** : quelle valeur de `max_len` choisis-tu, et pourquoi pas 171 ?

*Ta réponse :*
Je choisis max_len = 33, car cette longueur permet de conserver la grande majorité des SMS tout en limitant les séquences trop longues. 
Je ne choisis pas 171, car cela correspond au message le plus long : adapter tous les messages à cette longueur serait inutilement coûteux en calcul et en mémoire, puisque la plupart des SMS sont bien plus courts.

## 3.d Lire les messages

🎯 Lire de vrais exemples : les chiffres ne disent pas tout. Repère ce qui distingue un spam (mots, chiffres, majuscules, numéros courts...).

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `df[df["label"] == 1]` | garder seulement les spams | — |
| `.sample(n, random_state=...)` | n lignes au hasard, reproductibles | `random_state=SEED` |
| `.str.lower().str.split().explode().value_counts()` | les mots les plus fréquents (en minuscules, un mot par ligne, puis compter) | `.head(20)` |
| `.str.contains(motif).sum()` | combien de textes contiennent un motif | — |

In [67]:
# ✍️ À toi
# 1. affiche 5 ham et 5 spams au hasard (texte complet : pd.set_option("display.max_colwidth", None))
# 2. affiche les 20 mots les plus fréquents des spams, puis ceux des ham
# 3. compte les messages qui contiennent "&lt;" : que remplace ce code, et dans quelle classe apparaît-il ?
# Afficher le texte complet
pd.set_option("display.max_colwidth", None)

SEED = 42
# Afficher 5 spams au hasard
df_clean[df_clean["label"] == 1].sample(n=5, random_state=SEED)
# Afficher 5 hams au hasard
df_clean[df_clean["label"] == 0].sample(n=5, random_state=SEED)

# 20 mots les plus fréquents dans les spams
spams = df_clean[df_clean["label"] == 1]["text"]
print("Mots fréquents dans les spams :",  spams.str.lower().str.split().explode().value_counts().head(20))

# 20 mots les plus fréquents dans les ham
hams = df_clean[df_clean["label"] == 0]["text"]
print("Mots fréquents dans les ham :", hams.str.lower().str.split().explode().value_counts().head(20))

df_clean.loc[df_clean["label"] == 1, "text"].str.contains("&lt;", case=False, na=False).sum()
# Compter les messages contenant &lt; dans chaque classe
for label, nom in [(0, "ham"), (1, "spam")]:
    nb = df_clean.loc[df_clean["label"] == label, "text"].str.contains(
        "&lt;", regex=False, na=False
    ).sum()
    print(f"{nom} : {nb} messages")



Mots fréquents dans les spams : text
to        594
a         330
call      298
your      241
you       226
for       182
the       181
free      157
or        157
2         151
is        138
on        119
ur        119
from      118
have      115
txt       114
and       104
text       96
u          96
mobile     95
Name: count, dtype: int64
Mots fréquents dans les ham : text
i       2069
you     1610
to      1471
the     1053
a        964
and      816
u        801
in       730
my       670
is       634
me       564
for      476
of       471
that     432
have     420
it       419
but      397
are      389
so       387
your     377
Name: count, dtype: int64
ham : 222 messages
spam : 0 messages


✍️ **À écrire (2-3 phrases)** : ce qui distingue un spam d'après tes lectures, et ce que tu fais de `&lt;#&gt;` (le garder tel quel, le remplacer ?).

*Ta réponse :*
&lt;#&gt;` apparaît dans des messages ham, et pas dans les spams. Le remplacer ne permettra donc pas, à lui seul, de mieux reconnaître les spams

---
# Étape 4 — Séparer train / validation / test

🎯 Trois paquets, avec **la même part de spams (12,63 %)** dans chacun :

```
df_clean : 5 169 SMS
┌──────────────────── 70 % train ────────────────────┬── 15 % val ──┬── 15 % test ──┐
│ le modèle apprend dessus                            │ on règle :   │ on ne le      │
│                                                     │ époques,     │ regarde qu'à  │
│                                                     │ seuil...     │ la toute fin  │
└─────────────────────────────────────────────────────┴──────────────┴───────────────┘
```

`train_test_split` ne coupe qu'en **deux** : il faut l'appeler **deux fois**.
1. couper `df_clean` en `train_val_df` (85 %) et `test_df` (15 %) ;
2. couper `train_val_df` en `train_df` et `val_df`. Attention : 15 % du total = **15 / 85** de `train_val_df`.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `train_test_split(df, ...)` | coupe un DataFrame en deux | `test_size=`, `stratify=` la colonne des classes, `random_state=SEED` |

Les colonnes `n_car` et `n_mots` peuvent partir avec : elles ne gênent pas.

In [68]:
# ✍️ À toi
# 1. train_val_df, test_df = 1re coupe (test = 15 %), stratifiée sur label
# 2. train_df, val_df = 2e coupe de train_val_df (val = 15 / 85), stratifiée sur label
# 3. affiche la taille et la part de spams de chaque paquet
# Étape 1 : 85 % pour train + validation, 15 % pour test
train_val_df, test_df = train_test_split(
    df_clean,
    test_size=0.15,
    stratify=df_clean["label"],
    random_state=SEED
)

# Étape 2 : parmi les 85 %, réserver 15 % du total à la validation
train_df, val_df = train_test_split(
    train_val_df,
    test_size=15/85,
    stratify=train_val_df["label"],
    random_state=SEED
)

print("Train :", len(train_df))
print("Validation :", len(val_df))
print("Test :", len(test_df))

Train : 3617
Validation : 776
Test : 776


In [69]:
# ✅ Contrôle
assert (len(train_df), len(val_df), len(test_df)) == (3617, 776, 776), (len(train_df), len(val_df), len(test_df))
for nom, paquet in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert round(paquet["label"].mean(), 4) == 0.1263, f"{nom} : stratification ratée"
assert not set(train_df["text"]) & set(test_df["text"]), "un même SMS est dans train ET test"
print("✅ étape 4")

✅ étape 4


---
# Étape 5 — Enregistrer pour les phases suivantes

🎯 Écrire les trois paquets dans `src/train.csv`, `src/val.csv`, `src/test.csv`, avec seulement `text` et `label`. Les templates 2 et 3 partiront de ces fichiers.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `.to_csv(chemin, index=False)` | écrit un CSV | `index=False` : sinon une colonne de numéros en plus |

Ces fichiers sont écrits en UTF-8 (le défaut de `to_csv`) : plus besoin de `encoding=` pour les relire.

In [70]:
# ✍️ À toi
# Pour chacun des 3 paquets : garder text et label, écrire dans DOSSIER_SRC / "<nom>.csv"
train_df[["text", "label"]].to_csv(DOSSIER_SRC / "train.csv", index=False)
val_df[["text", "label"]].to_csv(DOSSIER_SRC / "val.csv", index=False)
test_df[["text", "label"]].to_csv(DOSSIER_SRC / "test.csv", index=False)


In [71]:
# ✅ Contrôle
for nom, n in [("train", 3617), ("val", 776), ("test", 776)]:
    relu = pd.read_csv(DOSSIER_SRC / f"{nom}.csv")
    assert list(relu.columns) == ["text", "label"], f"{nom}.csv : colonnes {list(relu.columns)}"
    assert len(relu) == n, f"{nom}.csv : {len(relu)} lignes au lieu de {n}"
print("✅ étape 5 : données prêtes pour les templates 2 et 3")

✅ étape 5 : données prêtes pour les templates 2 et 3


---
# 📝 Pour le livrable

Le notebook final doit **expliquer** le preprocessing. Garde de côté, en une ligne chacun :

| Décision | Ta justification |
|---|---|
| encodage latin-1 | |
| 50 messages recollés | |
| 403 doublons supprimés avant le split | |
| métrique principale (pas l'accuracy) | |
| `max_len` choisi | |
| 70 / 15 / 15 stratifié | |
| `&lt;#&gt;` | |